In [1]:
import os
import pandas as pd
import glob

In [ ]:
# Path to the Prokka annotation output directory
# Expected structure: annotations/prokka_plasmids/<sample_id>/<plasmid_id>/<annotation>.tsv
base_dir = 'annotations/prokka_plasmids/'

# Glob pattern locates all TSV annotation files two directory levels below the base
search_pattern = os.path.join(base_dir, "*", "*", "*.tsv")
tsv_files = glob.glob(search_pattern)

print(f"Found {len(tsv_files)} annotation files.")

all_genes = []

for f in tsv_files:
    try:
        # Extract sample and plasmid identifiers from the file path
        parts = f.split(os.sep)
        sample_id = parts[-3]
        plasmid_name = parts[-2]
        
        df = pd.read_csv(f, sep='\t')
        
        # Retain only the columns relevant to downstream gene content analysis
        df = df[['locus_tag', 'ftype', 'gene', 'product']]
        
        # Restrict to protein-coding sequences (CDS)
        df = df[df['ftype'] == 'CDS']
        
        df['Sample'] = sample_id
        df['Plasmid_ID'] = plasmid_name
        
        all_genes.append(df)
        
    except Exception as e:
        print(f"Error parsing {f}: {e}")

master_gene_table = pd.concat(all_genes, ignore_index=True)

# Flag genes whose product description contains stress- or resistance-related keywords
keywords = ['heat', 'cold', 'osmotic', 'betaine', 'proline', 'arsenic', 'copper', 'mercury', 'resistance', 'toxin']
pattern = '|'.join(keywords)
master_gene_table['is_interesting'] = master_gene_table['product'].str.contains(pattern, case=False, na=False)

master_gene_table.to_csv('plasmid_gene_inventory.csv', index=False)

print("Parsed all plasmids. Saved to 'plasmid_gene_inventory.csv'.")
print("\nPreview of potentially interesting genes:")